# EcoSentinel: TimesFM Zero-Shot Forecasting
## Can Google's pre-trained forecasting model beat our LSTM without any training?

**What this notebook does, top to bottom:**
1. Clones the EcoSentinel project and installs Google's `timesfm` package
2. Loads the **exact 140-day test set** our LSTM was graded on
   (`data/eval_results/forecast_test_set.npz`), plus the LSTM's saved scores
3. Downloads **TimesFM 2.5** (200M parameters, Apache-2.0) from Hugging Face
4. Forecasts each test day from the previous 30 days of PM2.5, with **no training**
   on air-quality data (that's what "zero-shot" means)
5. Scores TimesFM against the LSTM and 3 baselines, overall and per city
6. Saves and downloads the results

**Fairness:** same 140 days, same 30-day history, same metrics as the LSTM.
One difference to remember: the LSTM sees PM2.5 + wind + temperature;
TimesFM here sees **PM2.5 only**.

**Runtime:** the default **CPU** runtime is enough. TimesFM isn't training,
only predicting 140 values.

In [ ]:
# ── Cell 1: Clone the repo and install TimesFM ───────────────────────────────
# Takes ~1 to 2 minutes. If you see 'already exists', restart the runtime instead
# (Runtime → Disconnect and delete runtime) so you get the latest code.

!git clone https://github.com/maahipatel05/ecosentinel.git
%cd ecosentinel

!pip install "timesfm[torch]" --quiet

print("\n✅ Installation complete")

---
## Step 1: Load the exam and the LSTM's scores

`forecast_test_set.npz` holds the exact test set the LSTM was graded on:

| Array | Shape | Meaning |
|---|---|---|
| `X_te` | (140, 30, 3) | 140 flashcards × 30 days × [PM2.5, wind, temperature] |
| `y_te` | (140,) | the real PM2.5 on each test day (the answers) |
| `city_te` | (140,) | which city each flashcard belongs to |
| `date_te` | (140,) | the day being predicted |
| `preds_lstm` | (140,) | the LSTM's prediction for each test day |

In [ ]:
# ── Cell 2: Load the test set and the LSTM's saved scores ────────────────────
# Takes <5 seconds.

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

TEST_SET_PATH    = Path("data/eval_results/forecast_test_set.npz")
LSTM_RESULTS     = Path("data/eval_results/forecast_results.json")
TFM_RESULTS_PATH = Path("data/eval_results/timesfm_zeroshot_results.json")
TFM_PREDS_PATH   = Path("data/eval_results/timesfm_zeroshot_preds.npz")

exam = np.load(TEST_SET_PATH)
X_te, y_te       = exam["X_te"], exam["y_te"]
city_te, date_te = exam["city_te"], exam["date_te"]
preds_lstm       = exam["preds_lstm"]
CITIES = ["Delhi", "Los Angeles", "Seoul", "Krakow"]

lstm_saved = json.loads(LSTM_RESULTS.read_text())

print(f"Test set: {X_te.shape[0]} days × {X_te.shape[1]} days of history × {X_te.shape[2]} features")
for city in CITIES:
    m = city_te == city
    print(f"  {city:<12} {int(m.sum())} days   {date_te[m][0]} to {date_te[m][-1]}")
lstm_overall = next(r for r in lstm_saved["overall"] if r["model"] == "LSTM (ours)")
print(f"\nLSTM overall MAE to beat: {lstm_overall['mae']:.2f} µg/m³")
print("\n✅ Exam loaded")

---
## Step 2: Load TimesFM 2.5

TimesFM reads history in **patches of 32 time points** and uses a **transformer**
(the architecture behind modern language models) to forecast.

Settings we use (`ForecastConfig`):

| Setting | Value | Why |
|---|---|---|
| `max_context` | 32 | our history is 30 days; must be a multiple of the 32-point patch, and shorter inputs are padded and **masked** (ignored) |
| `max_horizon` | 128 | we only need 1 day ahead; 128 is the model's output patch size |
| `normalize_inputs` | True | rescales each series internally, so Delhi's 300s and LA's 10s are handled the same way |
| `use_continuous_quantile_head` | True | gives a proper uncertainty range (10th to 90th percentile) |
| `infer_is_positive` | True | PM2.5 can't be negative, so forecasts aren't allowed to be |
| `fix_quantile_crossing` | True | keeps the 10th percentile below the 20th, and so on |

The first run downloads the model (~1 GB) from Hugging Face. That takes a few minutes.

In [ ]:
# ── Cell 3: Download and set up TimesFM 2.5 ──────────────────────────────────
# First run: a few minutes (model download). Re-runs: seconds.

import torch
import timesfm

torch.manual_seed(42)

t0 = time.time()
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False,  # compiling only pays off for long or repeated runs
)
model.compile(
    timesfm.ForecastConfig(
        max_context=32,
        max_horizon=128,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
n_params = sum(p.numel() for p in model.model.parameters())
print(f"✅ TimesFM 2.5 ready in {time.time() - t0:.0f}s  ({n_params / 1e6:.0f}M parameters)")

---
## Step 3: Forecast every test day (zero-shot)

For each of the 140 test days we give TimesFM **only the 30 previous days of PM2.5**
and ask for 1 day ahead. No training, no wind, no temperature.

TimesFM returns two things:
- a **point forecast** (its single best guess), and
- **quantiles**: a range. We keep the 10th and 90th percentiles, an **80% interval**
  ("80% sure the real value is between these two numbers").

In [ ]:
# ── Cell 4: Zero-shot forecasts for all 140 test days ────────────────────────
# Takes well under a minute on CPU.

contexts = [X_te[i, :, 0].astype(np.float32) for i in range(len(X_te))]  # PM2.5 only

t0 = time.time()
point, quantiles = model.forecast(horizon=1, inputs=list(contexts))
elapsed = time.time() - t0

# quantiles[..., k]: k=0 is the mean, k=1..9 are the 10th..90th percentiles
preds_tfm = point[:, 0]
q10, q90  = quantiles[:, 0, 1], quantiles[:, 0, 9]

print(f"Forecast {len(preds_tfm)} days in {elapsed:.1f}s")
print("\nFirst 3 Delhi test days:")
d = np.where(city_te == "Delhi")[0][:3]
for i in d:
    print(f"  {date_te[i]}  real {y_te[i]:6.1f}  TimesFM {preds_tfm[i]:6.1f}  "
          f"(80% range {q10[i]:6.1f} to {q90[i]:6.1f})  LSTM {preds_lstm[i]:6.1f}")

---
## Step 4: Score everyone on the same exam

Same metrics as the LSTM notebook (lower is better):
- **MAE**: average miss in µg/m³
- **RMSE**: like MAE, but big misses count more
- **MAPE**: average miss as a percentage

Plus one new number for TimesFM: **80% interval coverage**, the share of test days
where the real value landed inside TimesFM's 10th to 90th percentile range.
A well-calibrated 80% range should contain the truth about 80% of the time.

In [ ]:
# ── Cell 5: Score all forecasters, overall and per city, and save ────────────
# Takes <5 seconds.

def compute_metrics(y_true, y_pred, label):
    y_true = np.array(y_true, dtype=np.float32)
    y_pred = np.clip(np.array(y_pred, dtype=np.float32), 0, None)
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100)
    return {"model": label, "mae": round(mae, 3), "rmse": round(rmse, 3), "mape": round(mape, 2)}


def print_table(rows, title):
    print(f"\n{title}")
    print(f"{'Model':<36}  {'MAE':>8}  {'RMSE':>8}  {'MAPE':>8}")
    print("-" * 66)
    for r in rows:
        print(f"{r['model']:<36}  {r['mae']:>8.2f}  {r['rmse']:>8.2f}  {r['mape']:>7.1f}%")


predictions = {
    "Persistence (same as today)":        X_te[:, -1, 0],
    "Seasonal naive (same day last week)": X_te[:, -7, 0],
    "7-day rolling average":              X_te[:, -7:, 0].mean(axis=1),
    "LSTM (ours)":                        preds_lstm,
    "TimesFM 2.5 (zero-shot)":            preds_tfm,
}

overall = [compute_metrics(y_te, p, name) for name, p in predictions.items()]

# Sanity check: baselines and LSTM must match the LSTM notebook's saved scores
saved = {r["model"]: r["mae"] for r in lstm_saved["overall"]}
for r in overall:
    if r["model"] in saved:
        assert abs(r["mae"] - saved[r["model"]]) < 0.01, f"mismatch for {r['model']}"
print("✅ Baselines and LSTM match the saved scores: same exam, fair comparison")

print_table(overall, f"OVERALL ({len(y_te)} test days across {len(CITIES)} cities)")

per_city, coverage = {}, {}
inside = (y_te >= q10) & (y_te <= q90)
for city in CITIES:
    m = city_te == city
    per_city[city] = [compute_metrics(y_te[m], p[m], name) for name, p in predictions.items()]
    coverage[city] = round(float(inside[m].mean() * 100), 1)
    print_table(per_city[city], f"{city} ({int(m.sum())} test days)")
coverage["overall"] = round(float(inside.mean() * 100), 1)

print("\nTimesFM 80% interval coverage (ideal ≈ 80%):")
for k, v in coverage.items():
    print(f"  {k:<12} {v:5.1f}%")

tfm  = next(r for r in overall if r["model"] == "TimesFM 2.5 (zero-shot)")
lstm = next(r for r in overall if r["model"] == "LSTM (ours)")
change = (lstm["mae"] - tfm["mae"]) / lstm["mae"] * 100
print(f"\n→ TimesFM MAE {tfm['mae']:.2f} vs LSTM {lstm['mae']:.2f}: "
      f"{abs(change):.1f}% {'lower' if change > 0 else 'higher'}")

results = {
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "config": {
        "model": "google/timesfm-2.5-200m-pytorch", "mode": "zero-shot",
        "inputs": "PM2.5 only, 30-day context", "horizon": 1,
        "test_set": str(TEST_SET_PATH), "test_days": int(len(y_te)),
        "timesfm_package": timesfm.__version__ if hasattr(timesfm, "__version__") else "see pip",
    },
    "overall": overall,
    "per_city": per_city,
    "timesfm_80pct_interval_coverage": coverage,
}
TFM_RESULTS_PATH.write_text(json.dumps(results, indent=2))
np.savez(TFM_PREDS_PATH, preds_tfm=preds_tfm, q10=q10, q90=q90, city_te=city_te, date_te=date_te)
print(f"\n✅ Saved scores → {TFM_RESULTS_PATH}")
print(f"✅ Saved predictions → {TFM_PREDS_PATH}")

In [ ]:
# ── Cell 6: Download the results ─────────────────────────────────────────────
# Move both files into ecosentinel/data/eval_results/ on your computer.

from google.colab import files
for path in (TFM_RESULTS_PATH, TFM_PREDS_PATH):
    files.download(str(path))
print("✅ Files downloading, check your browser's downloads folder.")